In [1]:
from google.colab import drive
drive.mount('/content/drive/')

# Cambia la directory di lavoro in quella del tuo progetto
# (Assicurati che il percorso sia esattamente quello in cui tieni la cartella eomt sul tuo Drive)
%cd /content/drive/MyDrive/project/semantic-segmentation-roads/eomt

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
/content/drive/.shortcut-targets-by-id/1kGi4cSNJjM14ClVvPXJ9VY70JDkofHGn/project/semantic-segmentation-roads/eomt


In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
import sys
import torch

PROJECT_DIR = "/content/drive/MyDrive/project/semantic-segmentation-roads"
EOMT_DIR = f"{PROJECT_DIR}/eomt"

state_dict_path_coco = "/content/drive/MyDrive/project/models_weights/eomt_coco.bin"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

os.chdir(PROJECT_DIR)

sys.path.insert(0, EOMT_DIR)     # for models.*, datasets.*, training.*
sys.path.insert(0, PROJECT_DIR)  # for eomt.*, utils.*

IMG_SIZE = (640,640)


from eomt.semantic_eval import evaluate_semantic
from utils.model_loading import get_config, build_model, load_weights
from utils.data_loading import build_datamodule

# --- 1. CONFIGURAZIONI ---
# Usiamo la configurazione di Cityscapes per l'architettura (così avrà 19 classi)
config = get_config()

# --- 2. INIZIALIZZAZIONE DEL DATASET (Cityscapes) ---
data = build_datamodule(config, batch_size=2, img_size=IMG_SIZE)



# --- 3. INIZIALIZZAZIONE DELL'ARCHITETTURA (Cityscapes) ---
model = build_model(config, img_size=IMG_SIZE, num_classes=data.num_classes, masked_attn_enabled=True)

# --- 4. CARICAMENTO DEI PESI DA COCO ---
model = load_weights(model, state_dict_path_coco, device).to(device)

print("\nModello e Dati pronti per il fine-tuning!")


In [4]:
def count_params(module, trainable_only=False):
    if trainable_only:
        return sum(p.numel() for p in module.parameters() if p.requires_grad)
    return sum(p.numel() for p in module.parameters())


def print_layer_param_counts(eomt_network):
    layer_names = [
        "mask_head.0",
        "mask_head.2",
        "mask_head.4",
        "upscale.0.conv1",
        "upscale.0.conv2",
        "upscale.1.conv1",
        "upscale.1.conv2",
        "class_head",
    ]

    print("Parametri dei layer originali:\n")

    total = 0

    for name in layer_names:
        module = eomt_network.get_submodule(name)
        n_params = count_params(module)
        total += n_params

        print(f"{name:20s} {type(module).__name__:20s} {n_params:,}")

    print(f"\nTotale layer stampati: {total:,}")


print_layer_param_counts(model.network)

Parametri dei layer originali:

mask_head.0          Linear               590,592
mask_head.2          Linear               590,592
mask_head.4          Linear               590,592
upscale.0.conv1      ConvTranspose2d      2,360,064
upscale.0.conv2      Conv2d               6,912
upscale.1.conv1      ConvTranspose2d      2,360,064
upscale.1.conv2      Conv2d               6,912
class_head           Linear               15,380

Totale layer stampati: 6,521,108


In [ ]:
# --- 1. CONGELAMENTO GLOBALE (FREEZING) ---
for param in model.parameters():
    param.requires_grad = False

# --- 2. SCONGELAMENTO DELLA PREDICTION HEAD ---
for param in model.network.class_head.parameters():
    param.requires_grad = True

for param in model.network.upscale.parameters():
  param.requires_grad = True

for param in model.network.mask_head.parameters():
  param.requires_grad = True

# --- 4. VERIFICA ---
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Parametri TOTALI del modello: {total_params:,}")
print(f"Parametri ADDESTRABILI (scongelati): {trainable_params:,}")
print(f"Percentuale di pesi in addestramento: {(trainable_params/total_params)*100:.4f}%")


Parametri TOTALI del modello: 93,498,644
Parametri ADDESTRABILI (scongelati): 6,524,180
Percentuale di pesi in addestramento: 6.9778%


In [6]:
import os
from datetime import datetime
import torch

max_epochs = 20

# -------------------------
# 0. Nome run e cartelle
# -------------------------
run_name = datetime.now().strftime(f"3_components_{max_epochs}ep")

ckpt_dir = f"/content/drive/MyDrive/project/checkpoints/{run_name}"
os.makedirs(ckpt_dir, exist_ok=True)

old_ckpt_dir = "/content/drive/MyDrive/project/checkpoints/3_components_20ep"
resume_ckpt_path = f"{old_ckpt_dir}/last.ckpt"
print("Checkpoint exists:", os.path.exists(resume_ckpt_path))
print(resume_ckpt_path)



Checkpoint exists: True
/content/drive/MyDrive/project/checkpoints/3_components_20ep/last.ckpt


In [ ]:
import wandb
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger



# 1. Eseguiamo il login a Weights & Biases
# (Ti chiederà la chiave API se non sei già loggata nella sessione corrente)
wandb.login()

# 2. Creiamo il logger: tutto verrà salvato sul tuo profilo nel progetto "eomt-cityscapes"
wandb_logger = WandbLogger(project="eomt-cityscapes", name=run_name, log_model=False)

# Log config utile su WandB
wandb_logger.experiment.config.update({
    "run_name": run_name,
    "max_epochs": max_epochs,
    # "real_epochs": real_epochs,
    "precision": "16-mixed",
    "batch_size": getattr(data, "batch_size", None),
    "img_size": getattr(data, "img_size", None),
    "num_classes": getattr(data, "num_classes", None),
    "finetuning": run_name,
    # "resume_from": resume_ckpt_path
})

# 3. Salvataggio dei pesi (checkpoint) su Drive
checkpoint_callback = ModelCheckpoint(
    dirpath=ckpt_dir,
    filename="best-{epoch:02d}",
    save_last=True,
    save_top_k=2,
    monitor="metrics/val_iou_all",
    mode="max"
)

# Log learning rate su WandB
lr_monitor = LearningRateMonitor(logging_interval="step")

print(f"Checkpoint directory: {ckpt_dir}")



print("Configurazione del PyTorch Lightning Trainer...")

trainer = pl.Trainer(
    max_epochs=max_epochs,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    logger=wandb_logger,
    callbacks=[checkpoint_callback, lr_monitor],

    log_every_n_steps=10,

    num_sanity_val_steps=2,

    enable_checkpointing=True,

    enable_progress_bar=True,

    # limit_train_batches=20,
    # limit_val_batches=5,
)

print("Avvio del Fine-Tuning!")
trainer.fit(model, datamodule=data
, ckpt_path=resume_ckpt_path)

# Alla fine dell'addestramento, diciamo a WandB che abbiamo finito
wandb.finish()
